# 06 — Fundamentals Cleaning (`fundamentals_clean.csv`)

## Purpose
Clean the quarterly company fundamentals and produce the `fundamentals` table from the data model.

## Grain
One row per company per fiscal quarter: primary key (`ticker`, `period_end`).

## Input
`data/raw/company_fundamentals.csv`. The raw file has one filing date **per metric**; the model needs one `filing_date` per row.

## Output
`data/processed/fundamentals_clean.csv`

| Column | Description |
|---|---|
| `ticker` | Project ticker |
| `period_end` | Last day of the fiscal quarter |
| `filing_date` | Day **all** numbers in the row were public (latest of the metric filing dates) |
| `revenue`, `net_income`, `total_assets`, `equity` | USD |
| `eps_diluted` | Diluted EPS in USD per share, as filed (not split-adjusted) |
| `profit_margin` | `net_income / revenue` |

## Rules
- `filing_date` is the **latest** filing date among the metrics that have a value in the row. This avoids look-ahead bias: a row is never used before every number in it was public.
- Rows with no metric values at all are dropped (they only carry filing dates for a period that has not been reported).
- Missing values stay missing. Nothing is estimated.
- Join to prices **by `filing_date`, never by `period_end`**: for each price date, use the latest row with `filing_date` on or before that date (for example `pd.merge_asof` by ticker).

## Expected gaps (not errors)
- **Fiscal Q4 rows** have balance-sheet values (`total_assets`, `equity`) but no `revenue`, `net_income` or `eps_diluted`, because those come from the annual report and cannot be derived from this file.
- **JPM** has no `revenue` (banks report differently), so its `profit_margin` is empty.
- **JNJ** has no `equity` for most of 2021-2022.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW_PATH = Path("../../data/raw/company_fundamentals.csv")
PROCESSED_DIR = Path("../../data/processed")
COMPANIES_PATH = PROCESSED_DIR / "companies.csv"      # optional: used for the foreign-key check
OUTPUT_PATH = PROCESSED_DIR / "fundamentals_clean.csv"

TICKERS = ["AAPL", "JPM", "MSFT", "NVDA", "JNJ", "XOM", "KO", "AMZN", "LLY", "TSLA"]
PROJECT_TICKERS = set(TICKERS)
END = pd.Timestamp("2025-12-31")                       # last date of the project period

METRICS = ["revenue", "net_income", "eps_diluted", "total_assets", "equity"]
FLOW_METRICS = ["revenue", "net_income", "eps_diluted"]      # missing in fiscal Q4 rows

## 1. Load and validate the structure

In [2]:
fund = pd.read_csv(RAW_PATH, dtype={"ticker": str})
print(f"Loaded {len(fund):,} rows, {fund['ticker'].nunique()} tickers")

required = {"ticker", "period_end", *METRICS, *[f"{m}_filing_date" for m in METRICS]}
missing = required - set(fund.columns)
if missing:
    raise ValueError(f"company_fundamentals.csv is missing columns: {sorted(missing)}")

fund["ticker"] = fund["ticker"].str.strip().str.upper()
fund["period_end"] = pd.to_datetime(fund["period_end"], errors="coerce").dt.normalize()
assert fund["period_end"].notna().all(), "unparseable period_end"
for m in METRICS:
    fund[m] = pd.to_numeric(fund[m], errors="coerce")
    fund[f"{m}_filing_date"] = pd.to_datetime(fund[f"{m}_filing_date"], errors="coerce").dt.normalize()

unknown = set(fund["ticker"]) - PROJECT_TICKERS
if unknown:
    raise ValueError(f"Tickers outside the project list: {sorted(unknown)}")

dups = fund.duplicated(["ticker", "period_end"], keep=False)
print("Duplicate (ticker, period_end) rows:", int(dups.sum()))
fund = fund.drop_duplicates(["ticker", "period_end"], keep="last")

# A value without a filing date would silently be ignored below, so stop if that happens.
for m in METRICS:
    bad = fund[m].notna() & fund[f"{m}_filing_date"].isna()
    if bad.any():
        raise ValueError(f"{m}: {int(bad.sum())} values have no filing date")

Loaded 194 rows, 10 tickers
Duplicate (ticker, period_end) rows: 0


## 2. Drop rows with no values

These rows only carry filing dates for periods that were not reported in the file (for example fiscal quarters that ended in late 2025 but were filed in 2026).

In [3]:
empty = fund[METRICS].isna().all(axis=1)
print("Rows with no metric values (dropped):")
display(fund.loc[empty, ["ticker", "period_end"] + [f"{m}_filing_date" for m in METRICS]])
fund = fund[~empty].copy()
print("Rows kept:", len(fund))

Rows with no metric values (dropped):


,ticker,period_end,revenue_filing_date,net_income_filing_date,eps_diluted_filing_date,total_assets_filing_date,equity_filing_date
19,AAPL,2025-12-27,2026-01-30,2026-01-30,2026-01-30,2026-01-30,2026-01-30
59,JNJ,2025-12-28,NaT,NaT,NaT,2026-02-11,2026-04-22


Rows kept: 192


## 3. One `filing_date` per row

Latest filing date among the metrics that have a value. Rows whose metrics were filed on different dates are listed: they show where the raw data mixes the original filing with a later one (for example a prior-year quarter re-reported in the following annual report).

In [4]:
dated = pd.concat([fund[f"{m}_filing_date"].where(fund[m].notna()) for m in METRICS], axis=1)
fund["filing_date"] = dated.max(axis=1)
fund["filing_date_spread_days"] = (dated.max(axis=1) - dated.min(axis=1)).dt.days

assert fund["filing_date"].notna().all()
assert (fund["filing_date"] >= fund["period_end"]).all(), "filing_date before period_end"

late = fund[fund["filing_date"] > END]
print("Rows filed after the project period (dropped):", len(late))
fund = fund[fund["filing_date"] <= END].copy()

differ = fund[fund["filing_date_spread_days"] > 0]
print(f"Rows whose metrics have different filing dates: {len(differ)}")
display(differ[["ticker", "period_end", "filing_date", "filing_date_spread_days"]
               + [f"{m}_filing_date" for m in METRICS]])

Rows filed after the project period (dropped): 0
Rows whose metrics have different filing dates: 3


,ticker,period_end,filing_date,filing_date_spread_days,revenue_filing_date,net_income_filing_date,eps_diluted_filing_date,total_assets_filing_date,equity_filing_date
47,JNJ,2023-01-01,2024-02-16,365,2024-02-16,2024-02-16,2024-02-16,2023-02-16,2023-07-31
51,JNJ,2023-12-31,2024-05-01,75,2024-02-16,2024-02-16,2024-02-16,2024-02-16,2024-05-01
55,JNJ,2024-12-29,2025-04-23,69,NaT,NaT,NaT,2025-02-13,2025-04-23


## 4. Profit margin and review items

`profit_margin` is recomputed as `net_income / revenue` (empty when either value is missing or revenue is zero) and compared with the raw column. Extreme margins are listed for review; they can be real one-off items (for example JNJ in 2023-Q3 booked a large gain on the Kenvue separation).

In [5]:
raw_margin = fund["profit_margin"] if "profit_margin" in fund.columns else None
fund["profit_margin"] = np.where(
    fund["revenue"].notna() & fund["net_income"].notna() & fund["revenue"].ne(0),
    fund["net_income"] / fund["revenue"],
    np.nan,
)
if raw_margin is not None:
    diff = (fund["profit_margin"] - pd.to_numeric(raw_margin, errors="coerce")).abs().max()
    print("Max difference from the raw profit_margin:", diff)

extreme = fund[fund["profit_margin"].abs() > 0.6]
print("Profit margins above 60% or below -60% (review):")
display(extreme[["ticker", "period_end", "net_income", "revenue", "profit_margin"]])

Max difference from the raw profit_margin: 9.71445146547012e-17
Profit margins above 60% or below -60% (review):


,ticker,period_end,net_income,revenue,profit_margin
50,JNJ,2023-10-01,2.602800e+10,2.135100e+10,1.219053


## 5. Missing values and coverage

In [6]:
print("Missing values per ticker and metric (expected gaps are described at the top):")
display(fund.groupby("ticker")[METRICS].agg(lambda s: int(s.isna().sum())))

year_end_rows = fund[fund[FLOW_METRICS].isna().all(axis=1)]
print(f"Fiscal Q4-style rows without revenue/net income/EPS: {len(year_end_rows)}")
display(pd.crosstab(fund["ticker"], fund["period_end"].dt.year, margins=True))

Missing values per ticker and metric (expected gaps are described at the top):


,revenue,net_income,eps_diluted,total_assets,equity
ticker,,,,,
AAPL,5,5,5,0,0
AMZN,4,4,4,0,0
JNJ,3,3,3,0,9
JPM,19,4,4,0,0
KO,4,4,4,0,0
LLY,4,4,4,0,0
MSFT,5,5,5,0,0
NVDA,5,5,5,0,0
TSLA,4,4,4,0,0


Fiscal Q4-style rows without revenue/net income/EPS: 42


period_end,2021,2022,2023,2024,2025,All
ticker,,,,,,
AAPL,4,4,4,4,3,19
AMZN,4,4,4,4,3,19
JNJ,4,4,5,4,3,20
JPM,4,4,4,4,3,19
KO,4,4,4,4,3,19
LLY,4,4,4,4,3,19
MSFT,4,4,4,4,3,19
NVDA,4,4,4,4,4,20
TSLA,4,4,4,4,3,19


## 6. Validate and export

In [7]:
export_columns = ["ticker", "period_end", "filing_date", "revenue", "net_income",
                  "eps_diluted", "total_assets", "equity", "profit_margin"]
out = fund[export_columns].sort_values(["ticker", "period_end"]).reset_index(drop=True)

assert not out.duplicated(["ticker", "period_end"]).any(), "primary key (ticker, period_end) is not unique"
assert out["ticker"].eq(out["ticker"].str.upper()).all()
assert out["ticker"].isin(PROJECT_TICKERS).all()
assert out["filing_date"].notna().all() and out["period_end"].notna().all()
assert (out["revenue"].dropna() > 0).all(), "revenue must be positive"
assert (out["total_assets"].dropna() > 0).all(), "total_assets must be positive"

if COMPANIES_PATH.exists():
    known = set(pd.read_csv(COMPANIES_PATH, usecols=["ticker"], dtype=str)["ticker"])
    assert set(out["ticker"]) <= known, sorted(set(out["ticker"]) - known)
    print("All tickers exist in companies.csv.")

for col in ["period_end", "filing_date"]:
    out[col] = out[col].dt.strftime("%Y-%m-%d")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(out):,} rows to {OUTPUT_PATH.resolve()}")
display(out.head(8))

All tickers exist in companies.csv.
Saved 192 rows to D:\DEPI\MarketPulse Graduation Project\data\processed\fundamentals_clean.csv


,ticker,period_end,filing_date,revenue,net_income,eps_diluted,total_assets,equity,profit_margin
0,AAPL,2021-03-27,2021-04-29,8.958400e+10,2.363000e+10,1.40,3.371580e+11,6.917800e+10,0.263775
1,AAPL,2021-06-26,2021-07-28,8.143400e+10,2.174400e+10,1.30,3.298400e+11,6.428000e+10,0.267014
2,AAPL,2021-09-25,2021-10-29,NaN,NaN,NaN,3.510020e+11,6.309000e+10,NaN
3,AAPL,2021-12-25,2022-01-28,1.239450e+11,3.463000e+10,2.10,3.811910e+11,7.193200e+10,0.279398
4,AAPL,2022-03-26,2022-04-29,9.727800e+10,2.501000e+10,1.52,3.506620e+11,6.739900e+10,0.257098
5,AAPL,2022-06-25,2022-07-29,8.295900e+10,1.944200e+10,1.20,3.363090e+11,5.810700e+10,0.234357
6,AAPL,2022-09-24,2022-10-28,NaN,NaN,NaN,3.527550e+11,5.067200e+10,NaN
7,AAPL,2022-12-31,2023-02-03,1.171540e+11,2.999800e+10,1.88,3.467470e+11,5.672700e+10,0.256056


## Completion checklist

- [ ] Rows whose metrics have different filing dates reviewed (the latest date is used on purpose).
- [ ] Extreme profit margins reviewed and understood as real one-off items.
- [ ] Expected gaps (fiscal Q4 flows, JPM revenue, JNJ equity) are documented for the team.
- [ ] Joined to prices by `filing_date`, never `period_end`.
- [ ] `fundamentals_clean.csv` shared with Person 4.